# v20 — CatBoost Optuna Hyperparameter Search

CatBoost was added to the stack in v16 but has never been tuned — it runs on near-default params.
This notebook runs 100 Optuna TPE trials to find optimal CatBoost hyperparameters.

**This is a standalone tuning notebook — not a pipeline run.**  
It uses a single 80/20 split for speed (not 5-fold CV). Each trial takes ~30–45 seconds.

**Output:** Best params printed at the end → paste into notebook 21's CONFIG.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])

import sys
sys.path.insert(0, "../../")

import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

from src.features.feature_engineering import engineer_features, DROP_COLS, SOLD_COLS, RENTAL_COLS
from src.features.oof_encoding import apply_global_oof_encodings

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Libraries loaded.")

/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [2]:
# Load and prepare data — same feature engineering as the pipeline
train_raw = pd.read_csv("../../data/raw/train.csv", low_memory=False)

street_freq_map = train_raw['street_name'].value_counts().to_dict()
train_fe, train_caps = engineer_features(train_raw, street_freq_map=street_freq_map)

y = train_raw['resale_price'].values
global_price_med = float(np.median(y))

encode_pairs = [
    ("town",           "town_median_price",           1),
    ("postal_sector",  "postal_sector_median_price",  1),
    ("flat_model",     "flat_model_median_price",     1),
    ("town_flat_type", "town_flat_type_median_price", 3),
    ("Tranc_Year",     "year_median_price",           1),
]

# Apply OOF encodings — test_fe=None since we only need train here
apply_global_oof_encodings(train_fe, y, train_fe.copy(), encode_pairs, global_price_med)

always_drop = (
    ['id', 'resale_price', 'postal', 'block', 'town_flat_type', 'num_rooms']
    + DROP_COLS + SOLD_COLS + RENTAL_COLS
)
X = train_fe.drop(columns=always_drop, errors='ignore')
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].astype(str)

# 80/20 split for fast per-trial evaluation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)

# Preprocessor shared across all trials
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_cols)
])
preprocessor.fit(X_train)
X_train_arr = preprocessor.transform(X_train)
X_val_arr   = preprocessor.transform(X_val)

print(f"Train: {X_train_arr.shape}  |  Val: {X_val_arr.shape}")
print(f"Features: {X_train_arr.shape[1]}")

Train: (120507, 78)  |  Val: (30127, 78)
Features: 78


In [3]:
def objective_catboost(trial):
    params = {
        'iterations':        trial.suggest_int('iterations', 500, 2000),
        'depth':             trial.suggest_int('depth', 4, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg':       trial.suggest_float('l2_leaf_reg', 0.5, 10.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 30),
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train_arr, y_train_log)
    preds = np.expm1(model.predict(X_val_arr))
    return np.sqrt(mean_squared_error(y_val, preds))

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
print("Running CatBoost Optuna search (100 trials × 80/20 split)...")
study_cat.optimize(objective_catboost, n_trials=100, show_progress_bar=True)

print(f'\nBest val RMSE: ${study_cat.best_value:,.0f}')
print('Best params:', study_cat.best_params)

Running CatBoost Optuna search (100 trials × 80/20 split)...


Best trial: 38. Best value: 21483.9: 100%|██████████| 100/100 [4:28:54<00:00, 161.35s/it]   


Best val RMSE: $21,484
Best params: {'iterations': 1994, 'depth': 10, 'learning_rate': 0.058494228734099055, 'l2_leaf_reg': 1.496229574040861, 'colsample_bylevel': 0.7696123828614903, 'min_child_samples': 20}


In [4]:
# Print ready-to-paste CONFIG block for notebook 21
p = study_cat.best_params
print("── Paste this into notebook 21 CONFIG ──")
print(f'"catboost": {{')
print(f'    "iterations": {p["iterations"]}, "depth": {p["depth"]},')
print(f'    "learning_rate": {p["learning_rate"]:.4f},')
print(f'    "l2_leaf_reg": {p["l2_leaf_reg"]:.4f}, "colsample_bylevel": {p["colsample_bylevel"]:.4f},')
print(f'    "min_child_samples": {p["min_child_samples"]},')
print(f'    "loss_function": "RMSE", "random_seed": 42, "verbose": 0,')
print(f'}},')

── Paste this into notebook 21 CONFIG ──
"catboost": {
    "iterations": 1994, "depth": 10,
    "learning_rate": 0.0585,
    "l2_leaf_reg": 1.4962, "colsample_bylevel": 0.7696,
    "min_child_samples": 20,
    "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
},
